<a href="https://colab.research.google.com/github/rhodes-byu/stat-486/blob/main/notebooks/06-imbalance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b></b></p>

# Handling Imbalanced Classification Problems

This tutorial explores the challenges posed by imbalanced datasets and demonstrates practical techniques for addressing class imbalance in machine learning classification problems.

**Key Topics:**
- Detecting and understanding class imbalance
- Why standard accuracy metrics fail with imbalanced data
- Resampling techniques: undersampling and oversampling (SMOTE)
- Class weighting and cost-sensitive learning
- Comparing multiple approaches using appropriate evaluation metrics

## Section 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score, roc_curve
)

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, NearMiss

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## Section 2: Load and Explore Imbalanced Data

We'll create a synthetic imbalanced dataset with overlapping classes to demonstrate the challenges of class imbalance.

In [ ]:
from sklearn.datasets import make_classification

# Create synthetic imbalanced dataset with overlapping classes
# This creates a more realistic scenario where class imbalance AND class overlap
# both challenge the baseline model
X, y = make_classification(
    n_samples=5000,           # Total samples
    n_features=25,            # More features for richer problem
    n_informative=15,         # Only 15 features are truly informative
    n_redundant=5,            # 5 redundant features
    weights=[0.95, 0.05],     # 95% class 0, 5% class 1 (severe imbalance)
    flip_y=0.05,              # 5% label noise to make it harder
    random_state=42
)

# Create DataFrame for inspection
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
df['target'] = y

print("Dataset shape:", df.shape)
print("Synthetic dataset with class overlap and label noise")
print("\nFirst few rows:")
print(df.head())
print("\nFeatures:")
print(f"Number of features: {X.shape[1]}")

In [ ]:
# Analyze class distribution
print("Class Distribution:")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    class_name = "Majority" if label == 0 else "Minority"
    percentage = count / len(y) * 100
    print(f"Class {label} ({class_name}): {count} samples ({percentage:.1f}%)")

class_0_count = (y == 0).sum()
class_1_count = (y == 1).sum()
class_0_pct = class_0_count / len(y) * 100
class_1_pct = class_1_count / len(y) * 100
imbalance_ratio = class_0_count / class_1_count

print(f"\nClass balance ratio:")
print(f"Class 0 (Majority): {class_0_count} samples ({class_0_pct:.1f}%)")
print(f"Class 1 (Minority): {class_1_count} samples ({class_1_pct:.1f}%)")
print(f"\nImbalance ratio: {imbalance_ratio:.1f}:1")
print("\n⚠️  This severe imbalance means the baseline model can achieve high accuracy by always predicting the majority class.")

# Visualize class distribution
fig, ax = plt.subplots(figsize=(8, 5))

# Bar plot with logarithmic scale
ax.bar(['Class 0 (Majority)', 'Class 1 (Minority)'], [class_0_count, class_1_count], color=['skyblue', 'salmon'])
ax.set_ylabel('Number of Samples (Log Scale)', fontsize=11)
ax.set_title('Synthetic Dataset: Class Distribution', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining set class distribution:")
print(f"  Class 0: {(y_train == 0).sum()} ({(y_train == 0).sum() / len(y_train) * 100:.1f}%)")
print(f"  Class 1: {(y_train == 1).sum()} ({(y_train == 1).sum() / len(y_train) * 100:.1f}%)")

## Section 3: Understanding Class Imbalance Problems

### What is Class Imbalance?

Class imbalance occurs when one class (the **majority class**) has significantly more samples than the other class (the **minority class**). In this dataset, we have a 95:5 split.

### Why is Class Imbalance Problematic?

1. **Biased Model Predictions**: Models tend to favor the majority class because correctly predicting it gives high accuracy by default.

2. **Poor Minority Class Performance**: The model often performs poorly on the minority class—the class we usually care most about (fraud detection, disease diagnosis, rare events).

3. **Misleading Accuracy Metric**: A model that always predicts the majority class achieves 95% accuracy without any predictive power!

4. **Unbalanced Learning**: During training, the model sees far more examples of the majority class, leading to underfitting on the minority class.

### Example: A Naive Baseline

Let's demonstrate the problem with a naive classifier that always predicts the majority class.

In [ ]:
# Create a naive classifier that always predicts the majority class
y_naive = np.zeros_like(y_test)  # Always predict class 0

print("Naive Classifier (Always Predict Majority Class):")
print(f"Accuracy: {accuracy_score(y_test, y_naive):.4f}")
print(f"\nThis 95% accuracy is USELESS—it predicts nothing interesting!")
print(f"It detects zero instances of the minority class.")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_naive))

## Section 4: Evaluation Metrics for Imbalanced Data

With imbalanced data, accuracy is misleading. Instead, use metrics that focus on the minority class performance:

### Key Metrics Explained:

- **Precision**: Of all positive predictions, how many were correct? (Focus: False Positives)
- **Recall (Sensitivity)**: Of all actual positives, how many did we find? (Focus: False Negatives)
- **F1-Score**: Harmonic mean of precision and recall (balance between both)
- **AUC-ROC**: Area under the Receiver Operating Characteristic curve (threshold-independent)
- **PR-AUC**: Precision-Recall Area Under Curve (focuses on minority class)

Let's create a helper function to evaluate models comprehensively.

In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    """
    Comprehensive evaluation function for imbalanced classification.
    """
    # Get predictions and probabilities
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp)
    
    # Print results
    print(f"\n{'=' * 60}")
    print(f"{model_name}")
    print(f"{'=' * 60}")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}  (out of predicted positives, how many are correct)")
    print(f"Recall:         {recall:.4f}  (out of actual positives, how many we find)")
    print(f"Specificity:    {specificity:.4f}  (out of actual negatives, how many we correctly identify)")
    print(f"F1-Score:       {f1:.4f}  (balance between precision & recall)")
    print(f"ROC-AUC:        {roc_auc:.4f}  (threshold-independent performance)")
    print(f"PR-AUC:         {pr_auc:.4f}  (focuses on minority class)")
    
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(f"                Predicted Negative | Predicted Positive")
    print(f"Actual Negative |    {tn:5d}          |    {fp:5d}")
    print(f"Actual Positive |    {fn:5d}          |    {tp:5d}")
    
    return {
        'accuracy': accuracy, 'precision': precision, 'recall': recall,
        'f1': f1, 'roc_auc': roc_auc, 'pr_auc': pr_auc, 'specificity': specificity
    }

### Baseline Model: Standard Logistic Regression

Let's train a standard logistic regression model on the imbalanced training data without any special handling.

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train baseline model
model_baseline = LogisticRegression(max_iter=1000, random_state=42)
model_baseline.fit(X_train_scaled, y_train)

# Evaluate baseline
baseline_metrics = evaluate_model(model_baseline, X_test_scaled, y_test, "Baseline Logistic Regression")

# Print dynamic insights based on actual metrics
accuracy = baseline_metrics['accuracy']
recall = baseline_metrics['recall']
print(f"\n   ⚠️  Accuracy: {accuracy:.1%} | Recall: {recall:.1%}")
if recall < 0.5:
    print(f"   The baseline model is missing most of the minority class instances!")
else:
    print(f"   Even with decent recall, the baseline still struggles with imbalance.")

## Section 5: Undersampling Techniques

**Undersampling** reduces the majority class to balance the dataset. This discards majority class data but can be faster to train.

### Approaches:
- **Random Undersampling**: Randomly remove majority class samples
- **NearMiss**: Intelligently select majority class samples near the decision boundary

### How NearMiss Works

NearMiss selects majority class samples based on their proximity to minority class samples, keeping instances that are "close" to the decision boundary in feature space. This helps the classifier learn a clearer distinction between classes.

**Three Versions Available:**

- **NearMiss-1** (default): Keeps majority samples with the smallest average distance to their k nearest minority neighbors → Focuses on majority samples near the boundary
- **NearMiss-2**: Keeps majority samples with the smallest average distance to their k farthest minority neighbors → Ensures majority class "surrounds" minority class evenly  
- **NearMiss-3**: Two-step process → Selects majority samples that are difficult to classify, near minority instances

**Key Advantage:** Unlike random undersampling, NearMiss preserves informative samples at the decision boundary rather than discarding random majority examples.

In [ ]:
### Random Undersampling

print("Original training data distribution:")
print(f"Class 0: {(y_train == 0).sum()}, Class 1: {(y_train == 1).sum()}")

# Apply random undersampling
undersampler = RandomUnderSampler(random_state=42)
X_train_undersampled, y_train_undersampled = undersampler.fit_resample(X_train_scaled, y_train)

print(f"\nAfter random undersampling:")
print(f"Class 0: {(y_train_undersampled == 0).sum()}, Class 1: {(y_train_undersampled == 1).sum()}")
print(f"Dataset size reduced from {len(y_train)} to {len(y_train_undersampled)} samples")

# Train model on undersampled data
model_undersample = LogisticRegression(max_iter=1000, random_state=42)
model_undersample.fit(X_train_undersampled, y_train_undersampled)

# Evaluate
undersample_metrics = evaluate_model(model_undersample, X_test_scaled, y_test, "Random Undersampling")

print("\n  Notice: Much higher recall (74%) compared to baseline!")
print("  Trade-off: Slightly lower accuracy, but we catch more minority cases")

In [ ]:
### NearMiss Undersampling

# Apply NearMiss undersampling (selects informative majority samples)
nearmiss = NearMiss(version=1, n_neighbors=3)
X_train_nearmiss, y_train_nearmiss = nearmiss.fit_resample(X_train_scaled, y_train)

print(f"After NearMiss undersampling:")
print(f"Class 0: {(y_train_nearmiss == 0).sum()}, Class 1: {(y_train_nearmiss == 1).sum()}")

# Train model
model_nearmiss = LogisticRegression(max_iter=1000, random_state=42)
model_nearmiss.fit(X_train_nearmiss, y_train_nearmiss)

# Evaluate
nearmiss_metrics = evaluate_model(model_nearmiss, X_test_scaled, y_test, "NearMiss Undersampling")

print("\n💡 NearMiss keeps majority samples closest to the decision boundary,")
print("   which are more informative for learning than random samples.")

## Section 6: Oversampling Techniques

**Oversampling** increases the minority class by duplicating or generating synthetic samples. Retains all training data but risks overfitting.

### Approaches:
- **Random Oversampling**: Randomly duplicate minority class samples
- **SMOTE** (Synthetic Minority Over-sampling Technique): Generate synthetic minority samples by interpolating between existing samples

In [ ]:
### Random Oversampling

print("Original training data distribution:")
print(f"Class 0: {(y_train == 0).sum()}, Class 1: {(y_train == 1).sum()}")

# Apply random oversampling
oversampler = RandomOverSampler(random_state=42)
X_train_oversampled, y_train_oversampled = oversampler.fit_resample(X_train_scaled, y_train)

print(f"\nAfter random oversampling:")
print(f"Class 0: {(y_train_oversampled == 0).sum()}, Class 1: {(y_train_oversampled == 1).sum()}")
print(f"Dataset size increased from {len(y_train)} to {len(y_train_oversampled)} samples")

# Train model
model_oversample = LogisticRegression(max_iter=1000, random_state=42)
model_oversample.fit(X_train_oversampled, y_train_oversampled)

# Evaluate
oversample_metrics = evaluate_model(model_oversample, X_test_scaled, y_test, "Random Oversampling")

print("\n⚠️  Random oversampling duplicates minority samples.")
print("   This can lead to overfitting because the same samples appear in both training and testing.")

In [ ]:
### SMOTE (Synthetic Minority Over-sampling Technique)

# SMOTE generates synthetic samples by interpolating between k-nearest neighbors in the minority class.
# This creates new, unique synthetic samples rather than duplicating existing ones.

print("Original training data distribution:")
print(f"Class 0: {(y_train == 0).sum()}, Class 1: {(y_train == 1).sum()}")

# Apply SMOTE
smote = SMOTE(k_neighbors=5, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"\nAfter SMOTE:")
print(f"Class 0: {(y_train_smote == 0).sum()}, Class 1: {(y_train_smote == 1).sum()}")
print(f"Dataset size increased from {len(y_train)} to {len(y_train_smote)} samples")
print(f"✓ New synthetic samples generated instead of duplicating originals")

# Train model
model_smote = LogisticRegression(max_iter=1000, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)

# Evaluate
smote_metrics = evaluate_model(model_smote, X_test_scaled, y_test, "SMOTE")

print("\n✓ SMOTE generally provides better generalization than random oversampling")
print("  because it creates diverse synthetic samples within the minority class space.")

## Section 7: Class Weighting in Models

**Class weighting** assigns different penalties to misclassifications of each class during training.
Models with `class_weight` parameter (like LogisticRegression) apply higher costs to minority class errors.

This avoids data manipulation and works directly with the original imbalanced data.

In [ ]:
### Approach 1: Automatic Class Weight Balancing

# The `class_weight='balanced'` parameter automatically calculates weights inversely proportional to class frequencies.
# For our 95:5 imbalance:
# - Class 0 weight: 1 / 0.95 ~ 1.05
# - Class 1 weight: 1 / 0.05 ~ 20

print("Using class_weight='balanced'")
print(f"Class 1 (minority) errors are weighted ~19x more heavily than Class 0")

# Train model with balanced class weights
model_weighted = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_weighted.fit(X_train_scaled, y_train)

# Evaluate
weighted_metrics = evaluate_model(model_weighted, X_test_scaled, y_test, "Balanced Class Weights")

print("\n✓ Works directly on original imbalanced data")
print("✓ No data duplication or removal needed")

## Section 8: Comparing All Approaches

Let's create a comprehensive comparison of all our approaches using multiple metrics.

In [ ]:
### Build Comparison DataFrame

# Organize all metrics
all_models = {
    'Baseline': baseline_metrics,
    'Undersampling': undersample_metrics,
    'NearMiss': nearmiss_metrics,
    'Random Oversample': oversample_metrics,
    'SMOTE': smote_metrics,
    'Balanced Weights': weighted_metrics
}

# Create comparison table
comparison_df = pd.DataFrame(all_models).T
comparison_df = comparison_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']]

print("\n" + "="*90)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*90)
print(comparison_df.round(4))

print("\n  ✓ Recall: catch minority class instances")
print("  ✓ F1-Score: balance precision and recall")
print("  ✓ PR-AUC: focuses on minority class performance")
print("  ✓ ROC-AUC: threshold-independent evaluation")

In [ ]:
### Visualization 1: Metric Comparison

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Comparison of Imbalance Handling Approaches', fontsize=16, fontweight='bold')

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
axes_flat = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    ax = axes_flat[idx]
    values = comparison_df[metric].sort_values(ascending=False)
    colors = ['green' if i > 0.5 else 'orange' if i > 0.3 else 'red' for i in values]
    
    ax.barh(range(len(values)), values.values, color=colors)
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(values.index, fontsize=9)
    ax.set_xlabel(metric.replace('_', ' ').title(), fontsize=10)
    ax.set_xlim(0, 1)
    
    # Add value labels
    for i, v in enumerate(values.values):
        ax.text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
### Visualization 2: Precision-Recall Curves

fig, ax = plt.subplots(figsize=(10, 7))

# Store models for plotting
models_dict = {
    'Baseline': (model_baseline, 'gray'),
    'Undersampling': (model_undersample, 'blue'),
    'NearMiss': (model_nearmiss, 'purple'),
    'Random Oversample': (model_oversample, 'orange'),
    'SMOTE': (model_smote, 'red'),
    'Balanced Weights': (model_weighted, 'green')
}

for name, (model, color) in models_dict.items():
    y_scores = model.predict_proba(X_test_scaled)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_scores)
    pr_auc = average_precision_score(y_test, y_scores)
    
    ax.plot(recall, precision, marker='o', label=f'{name} (AP={pr_auc:.3f})', 
            color=color, linewidth=2, markersize=4)

ax.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves: Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.show()

print("Precision-Recall curves focus on minority class performance.")
print("Higher and rightward curves indicate better performance on minority class detection.")

In [ ]:
### Visualization 3: ROC Curves

fig, ax = plt.subplots(figsize=(10, 7))

models_dict = {
    'Baseline': (model_baseline, 'gray'),
    'Undersampling': (model_undersample, 'blue'),
    'NearMiss': (model_nearmiss, 'purple'),
    'Random Oversample': (model_oversample, 'orange'),
    'SMOTE': (model_smote, 'red'),
    'Balanced Weights': (model_weighted, 'green')
}

for name, (model, color) in models_dict.items():
    y_scores = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_auc = roc_auc_score(y_test, y_scores)
    
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})', 
            color=color, linewidth=2, marker='o', markersize=4)

# Plot diagonal (random classifier)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC=0.500)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves: Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.show()

print("ROC curves show trade-off between True Positive Rate and False Positive Rate.")
print("Higher curves indicate better overall classification performance.")

## Key Takeaways

### When to Use Each Technique:

1. **Undersampling (Random)**
   - ✓ Fast, simple, reduces training time
   - ✗ Loses majority class information
   - Best for: Large datasets with extreme imbalance

2. **Undersampling (NearMiss)**
   - ✓ Intelligently selects informative samples
   - ✓ Retains decision boundary information
   - ✗ More computationally expensive
   - Best for: Moderate imbalance, smaller datasets

3. **Random Oversampling**
   - ✓ Retains all original data
   - ✗ Creates duplicates → overfitting risk
   - Best for: When you must use all training data

4. **SMOTE**
   - ✓ Creates diverse synthetic samples
   - ✓ Prevents exact duplication
   - ✗ Can generate samples in noisy regions
   - Best for: Most real-world imbalanced problems (recommended!)

5. **Class Weighting (Balanced)**
   - ✓ No data manipulation
   - ✓ Simple to implement
   - ✓ Fast to train
   - Best for: Quick baseline, many algorithms support it

6. **Custom Class Weighting**
   - ✓ Fine-tune based on business requirements
   - ✓ Control precision-recall trade-off
   - Best for: When costs of errors differ significantly

### Decision Tree:

```
Do you have EXTREME imbalance (>100:1)?
├─ YES → Try Undersampling (NearMiss) or class weighting first
└─ NO → Try SMOTE or balanced class weights

Is training speed critical?
├─ YES → Use class weighting
└─ NO → Use SMOTE (generally best results)

Can you afford to lose data?
├─ YES → Try undersampling
└─ NO → Use oversampling or class weighting
```

### Most Important Metrics for Imbalanced Data:

| Metric | When to Use | Interpretation |
|--------|------------|-----------------|
| **Recall** | Always | Percentage of minority cases found |
| **Precision** | When false positives are costly | Percentage of predictions that are correct |
| **F1-Score** | Good overall balance metric | Harmonic mean of precision & recall |
| **ROC-AUC** | When threshold tuning is needed | Overall classification ability |
| **PR-AUC** | Best for imbalanced data | Focuses on minority class |

❌ **NEVER use accuracy alone** for imbalanced datasets!

✅ **ALWAYS check recall and F1-score** alongside other metrics

## Summary and Best Practices

### Quick Reference: Imbalance Handling Approaches

| Approach | Data Change | Training Time | Overfitting Risk | Ease of Use |
|----------|-------------|---------------|------------------|-------------|
| Baseline | None | ⭐ Fast | Medium | ⭐⭐⭐ |
| Random Undersampling | Discard 95% | ⭐ Very Fast | Low | ⭐⭐⭐ |
| NearMiss | Discard 95% | ⭐⭐ Fast | Low | ⭐⭐ |
| Random Oversampling | Duplicate | ⭐⭐ Medium | High | ⭐⭐⭐ |
| SMOTE | Synthetic | ⭐⭐ Medium | Medium | ⭐⭐ |
| Balanced Weights | None | ⭐⭐ Medium | Low | ⭐⭐⭐ |
| Custom Weights | None | ⭐⭐ Medium | Low | ⭐⭐ |

### What We Learned:

1. **Class imbalance breaks standard accuracy metrics** — a 95% accurate model that predicts the majority class for everything is useless

2. **Resampling techniques** (undersampling, SMOTE, oversampling) change the training data distribution

3. **Class weighting** adjusts the cost of errors during training without modifying data

4. **No one-size-fits-all solution** — the best approach depends on your data, computational resources, and business requirements

5. **Always evaluate using appropriate metrics** — Recall, F1, PR-AUC, and ROC-AUC are critical for imbalanced datasets

### Recommended Workflow:

1. **Identify imbalance**: Calculate class distribution and imbalance ratio
2. **Set baseline metrics**: Train initial model and measure using F1, Recall, PR-AUC
3. **Choose approach**: Start with SMOTE or balanced class weights
4. **Tune parameters**: Adjust weights or resampling ratios based on results
5. **Evaluate rigorously**: Use multiple metrics and cross-validation
6. **Validate on test set**: Ensure improvements generalize beyond training data